# Test Symmetric Dirichlet Parametrization using MeshFEM Variant Settings

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark, flip_avoiding_step_length
import energy

import numpy as np
import time, copy
import param_utils
import helper_funcs
from helper_funcs import HessianStats
import igl

import psutil
import parallelism
parallelism.set_max_num_tbb_threads(psutil.cpu_count(logical=True))

## User Settings

In [ ]:
import MeshFEMParamSolverEnum
MeshFEMParamSolverEnum.optionNamesFromIndex(17)

In [ ]:
ProjectionStrategy = ['Adaptive', 'Always'][1]
EigenvalueModification = 'Clamp' # Abs
ProjectionType = 'FBased' # XBased
AutodiffSetting = 'NoAD' # AD
SteplengthComputer = 'FlipAvoid' # NoFlipAvoid

In [ ]:
# Configure to match CM
usr_hessian_shift, usr_element_hessian_shift = 0.0, 1e-6
flip_avoid_backoff_factor = 0.8

## Read Mesh and Opt

In [ ]:
mesh_file_path = '../../models/Superman_cut3.msh.xz'
# mesh_file_path = '../../models/Lucy_3cuts.msh.xz'
m = helper_funcs.read_mesh(mesh_file_path)

In [ ]:
model_name = os.path.basename(mesh_file_path).split('.')[0]

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
# Tutte Initialization
uv_init = parametrization.harmonic(m, bdry_uv)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)

uv.setVars(uv_init.ravel())

In [ ]:
obj_history = []
time_histroy = []
grad_norm_history = []

hessian_projected_history = []
hessian_shifted_amount_history = []

step_norm_history = []
directional_derivative_history = []
uvs = []

def customCallback(prob, cb_ind):
    if cb_ind > 1:
        hessian_projected_history.append(int(prob.hessianWasProjected))
        hessian_shifted_amount_history.append(prob.lastFactorizationShiftMagnitude)
    it_time = time.perf_counter()
    obj_history.append(prob.energy())
    time_histroy.append(it_time)
    grad_norm_history.append(np.linalg.norm(prob.gradient()))
    uvs.append(uv.getVars())

def customSaveStepDCallback(prob, step, directional_derivative):
    step_norm_history.append(np.linalg.norm(step))
    directional_derivative_history.append(-directional_derivative)

In [ ]:
# Configuring User Options 
# AutodiffSetting
if AutodiffSetting == 'AD':  symmdiri_energy = energy.SymmetricDirichletDerivativeFree(2)
elif AutodiffSetting == 'NoAD': symmdiri_energy = energy.SymmetricDirichlet(2)
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Autodiff Setting {AutodiffSetting} is not implemented.")

# EigenvalueModification
if EigenvalueModification == 'Clamp': symmdiri_energy.useAbsProjection = False
elif EigenvalueModification == 'Abs': symmdiri_energy.useAbsProjection = True
else: raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Eigenvalue Modification {EigenvalueModification} is not implemented.")

# Construct `SymmetricDirichlet` parametrization energy and problem
param = mesh_energy.Parametrization(m, uv, symmdiri_energy)
param.elementHessianShift = usr_element_hessian_shift
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])

# Step Length Computer
if SteplengthComputer == 'FlipAvoid':
    prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
    prob.initialFeasibleStepLengthComputer.backoffFactor = flip_avoid_backoff_factor    # in accordance to Composite Majorization
elif SteplengthComputer == 'NoFlipAvoid':
    pass
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Steplength Computer {SteplengthComputer} is not implemented.")

prob.setCustomIterationCallback(customCallback)
prob.setCustomLineSearchBeganCallback(customSaveStepDCallback)

# Projection Type
if ProjectionType == 'FBased':  param.useXBasedProjection = False
elif ProjectionType == 'XBased': param.useXBasedProjection = True
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Projection Type {ProjectionType} is not implemented.")

In [ ]:
print("Initialization Done, Energy: ", prob.energy())

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = usr_hessian_shift
opt = prob.optimizer()

In [ ]:
opt.options.verboseNonPosDef = True
opt.options.niter = 200
# Projection Strategy
if ProjectionStrategy == 'Adaptive':
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
    opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
elif ProjectionStrategy == 'Always':
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
elif ProjectionStrategy == 'Never':
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
else:  raise NameError(f"[runSYDParam] MeshFEM Solver Configuration: Projection Strategy {ProjectionStrategy} is not implemented.")

In [ ]:
benchmark.reset()
start_time = time.time()
cr = opt.optimize()
benchmark.report()

In [ ]:
hessian_projected_history.append(int(prob.hessianWasProjected)) # The projection status of the Hessian used in are i-1
hessian_shifted_amount_history.append(prob.lastFactorizationShiftMagnitude)
# Also saving hessian_related information (arrays)
hessian_projected_arr = np.array(hessian_projected_history, dtype=int)
hessian_shifted_arr = np.array(hessian_shifted_amount_history, dtype=float)
hessian_indef_arr = np.array(cr.indefinite, dtype=int)

## Convergence Plot

In [ ]:
import matplotlib
from matplotlib import pyplot as plt

In [ ]:
# cm_gnorm = np.load('/Users/jpanetta/Downloads/UVs/grad_norm_history.npy')

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(grad_norm_history, label='MeshFEM')
# plt.plot(cm_gnorm, label='CM')

plt.xlabel('Iter', fontsize=12)
plt.ylabel('Grad Norm', fontsize=12)
plt.title(f"Model: {model_name}, H shift: {usr_hessian_shift}, EH shift: {usr_element_hessian_shift}, FA_backoff: {flip_avoid_backoff_factor}", fontsize=10)
plt.yscale("log")

plt.grid(linestyle='--', linewidth=0.5)
plt.legend()
# plt.xlim(0, 100)
plt.savefig(f'comparison_{model_name}_{usr_hessian_shift}_{usr_element_hessian_shift}_{flip_avoid_backoff_factor}.png')